## Train Generative Retrieval Model (T5)

Trains the T5 seq2seq model to predict the next item's Semantic ID from a user's interaction history.

**What this notebook does:**
1. Load Semantic ID dataset
2. Initialize T5 (~14.5M params)
3. Train with LR warmup + inverse sqrt decay (100k steps)
4. Evaluate with beam search → Recall@K, NDCG@K

In [1]:
import sys

if "../" not in sys.path:
    sys.path.insert(0, "../")

from pathlib import Path
from functools import partial

import torch
from torch.utils.data import DataLoader

from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

from tiger.dataset import TigerDataset, custom_collate
from tiger.model import create_model
from tiger.utils import get_device, set_seed
from tiger.evaluation import compute_metrics

In [2]:
# Paths
DATA_DIR = Path("../data/2014/processed")
SPLITS_PATH = DATA_DIR / "splits.parquet"
SEMANTIC_IDS_PATH = Path("../checkpoints/rqvae/semantic_ids.pt")
OUTPUT_DIR = Path("../checkpoints/train_eval_test")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset
SLIDING_WINDOW = True
MAX_SEQ_LEN = 20

# Model architecture
D_MODEL = 384
D_KV = 64
D_FF = 1024
NUM_LAYERS = 4
NUM_HEADS = 6
DROPOUT_RATE = 0.1

# Training
BATCH_SIZE = 256
EVAL_BATCH_SIZE = 64
NUM_STEPS = 100_000
LEARNING_RATE = 3e-4
WARMUP_STEPS = 10_000
EVAL_EVERY = 5_000
SAVE_EVERY = EVAL_EVERY

# Inference
BEAM_SIZE = 20

SEED = 42

In [3]:
set_seed(SEED)
device = get_device()

2026-09-20 19:17:12.719 | INFO     | tiger.utils:set_seed:27 - Random seed set to 42
2026-09-20 19:17:12.732 | INFO     | tiger.utils:get_device:16 - Using device: mps


In [4]:
sid_data = torch.load(SEMANTIC_IDS_PATH, weights_only=False)
sid_to_asin = sid_data["sid_to_asin"]

print(f"Items with Semantic IDs: {len(sid_to_asin):,}")

Items with Semantic IDs: 11,924


### Load Dataset

Two dataloaders: train (with sliding window) and validation (single target).

Train uses sliding window to create ~95k examples from 19k users by using all sub-prefixes
of each user's history. Val always uses the full training history to predict the held-out val item.

In [5]:
train_dataset = TigerDataset(
    splits_path=SPLITS_PATH,
    semantic_ids_path=SEMANTIC_IDS_PATH,
    split="train",
    max_seq_len=MAX_SEQ_LEN,
    sliding_window=SLIDING_WINDOW,
)

val_dataset = TigerDataset(
    splits_path=SPLITS_PATH,
    semantic_ids_path=SEMANTIC_IDS_PATH,
    split="val",
    max_seq_len=MAX_SEQ_LEN,
)

collate_fn = partial(custom_collate, pad_token_id=train_dataset.pad_token)


print(f"Train samples: {len(train_dataset):_}")
print(f"Val samples: {len(val_dataset):_}")
print(f"Vocab size: {train_dataset.vocab_size:_}")
print(f"Pad token: {train_dataset.pad_token}")

Train samples: 109_361
Val samples: 19_412
Vocab size: 3_026
Pad token: 3024


### Test Memorisation

In [ ]:
from torch.utils.data import DataLoader, Subset

# Make the selection repeatable.
sample_generator = torch.Generator().manual_seed(42)
sample_indices = torch.randperm(
    len(train_dataset),
    generator=sample_generator,
)[:32].tolist()

tiny_dataset = Subset(train_dataset, sample_indices)

# Put all 32 examples into one batch.
tiny_loader = DataLoader(
    tiny_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
)

tiny_batch = next(iter(tiny_loader))
tiny_batch = {
    name: tensor.to(device)
    for name, tensor in tiny_batch.items()
}

# Start a separate model from scratch.
set_seed(42)

probe_model = create_model(
    vocab_size=train_dataset.vocab_size,
    pad_token_id=train_dataset.pad_token,
    d_model=D_MODEL,
    d_kv=D_KV,
    d_ff=D_FF,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    dropout_rate=DROPOUT_RATE,
).to(device)

probe_optimizer = torch.optim.AdamW(
    probe_model.parameters(),
    lr=3e-4,  # Same number as 0.0003.
)

for step in range(1, 201):
    probe_model.train()
    probe_optimizer.zero_grad(set_to_none=True)

    loss = probe_model(**tiny_batch).loss
    loss.backward()

    # Match the gradient clipping used by your Trainer.
    # This caps unusually large gradients.
    torch.nn.utils.clip_grad_norm_(
        probe_model.parameters(),
        max_norm=1.0,
    )

    probe_optimizer.step()

    if step % 25 == 0:
        probe_model.eval()

        with torch.no_grad():
            eval_loss = probe_model(**tiny_batch).loss.item()

            generated = probe_model.generate(
                input_ids=tiny_batch["input_ids"],
                attention_mask=tiny_batch["attention_mask"],
                num_beams=1,
                do_sample=False,
                max_new_tokens=4,
            )

            # Remove the decoder's initial START token.
            predicted_ids = generated[:, 1:5]

            exact_accuracy = (
                predicted_ids == tiny_batch["labels"]
            ).all(dim=1).float().mean().item()

        print(
            f"Step {step:3d} | "
            f"Loss: {eval_loss:.4f} | "
            f"Exact item accuracy: {exact_accuracy:.1%}"
        )

### Create Model

T5ForConditionalGeneration (random weights). Architecture matches paper:
- 4 encoder layers, 4 decoder layers
- 6 attention heads, 64 dim per head (d_model=384)
- d_ff=1024, dropout=0.1
- ~14.5M parameters

In [6]:
model = create_model(
    vocab_size=train_dataset.vocab_size,
    pad_token_id=train_dataset.pad_token,
    d_model=D_MODEL,
    d_kv=D_KV,
    d_ff=D_FF,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    dropout_rate=DROPOUT_RATE,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:_}")

Total parameters: 14_540_160


In [7]:
model.generation_config.num_beams = BEAM_SIZE
model.generation_config.num_return_sequences = BEAM_SIZE
model.generation_config.do_sample = False
model.generation_config.early_stopping = False
model.generation_config.max_length = 1 + train_dataset.num_levels
model.generation_config.max_new_tokens = None

### Optimizer and LR scheduler

In [8]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)


def inv_sqrt_schedule(step: int) -> float:
    if step < WARMUP_STEPS:
        return 1.0
    return (WARMUP_STEPS / step) ** 0.5


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=inv_sqrt_schedule)

### Training loop

- Train for 100k steps. 
- The training is step-based (not epoch-based). 
- We cycle through the dataloader indefinitely. 
- Every EVAL_EVERY steps, compute validation loss to monitor overfitting.

In [9]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=NUM_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,

    eval_strategy="steps",
    eval_steps=EVAL_EVERY,
    predict_with_generate=True,

    save_strategy="steps",
    save_steps=SAVE_EVERY,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_recall@10",
    greater_is_better=True,

    logging_steps=100,
    train_sampling_strategy="group_by_length",
    remove_unused_columns=False,
    dataloader_pin_memory=device.type == "cuda",
    seed=SEED,
    report_to="none",
)

In [10]:
retrieval_metrics_fn = partial(
    compute_metrics,
    num_levels=train_dataset.num_levels,
    beam_size=BEAM_SIZE,
    codebook_size=train_dataset.codebook_size,
    sid_to_asin=sid_to_asin,
    at_k=(5, 10),
)

In [11]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    compute_metrics=retrieval_metrics_fn,
    optimizers=(optimizer, scheduler),
)

In [15]:
from torch.utils.data import Subset

check_dataset = Subset(val_dataset, range(129))
results = trainer.evaluate(eval_dataset=check_dataset)

for name in (
    "eval_loss",
    "eval_recall@5",
    "eval_recall@10",
    "eval_ndcg@5",
    "eval_ndcg@10",
):
    print(f"{name}: {results[name]:.4f}")

Training Loss,Validation Loss,Step,Recall@5,Recall@10,Ndcg@5,Ndcg@10
No log,8.615322,0,0.000000,0.000000,0.000000,0.000000


eval_loss: 8.6153
eval_recall@5: 0.0000
eval_recall@10: 0.0000
eval_ndcg@5: 0.0000
eval_ndcg@10: 0.0000


In [ ]:
trainer.train()

print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation Recall@10:", trainer.state.best_metric)